# Random Forest classifier

Trains and evaluates the Random Forest model for pyrite deposit-type classification.

**Input data:** place the standardized Excel file at `../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx`.

**Note:** outputs were cleared for GitHub readability and reproducibility.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Load data
df = pd.read_excel('../data/processed/Pyrite_Standarized_data_file_New_Paper.xlsx')

# Define features and target
X = df[['Co', 'Ni', 'Cu', 'Zn', 'Se', 'Ag', 'Sb', 'Pb', 'Bi', 'As']]
y = df['Deposit type']

# Encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X_imputed, y_encoded, test_size=0.4, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Function to evaluate and report metrics including AUC
def train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, technique_name):
    print(f'\n==== {technique_name} ====')
    
    # Train model
    rf = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)
    rf.fit(X_train, y_train)
    
    # Validation metrics
    y_val_pred = rf.predict(X_val)
    y_val_proba = rf.predict_proba(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_auc = roc_auc_score(y_val, y_val_proba, multi_class='ovr')
    
    print(f'\nValidation Accuracy: {val_accuracy * 100:.2f}%')
    print(f'Validation AUC (OvR): {val_auc:.4f}')
    print('Validation Confusion Matrix:\n', confusion_matrix(y_val, y_val_pred))
    print('Validation Classification Report:\n', classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))
    
    # Test metrics
    y_test_pred = rf.predict(X_test)
    y_test_proba = rf.predict_proba(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr')

    print(f'\nTest Accuracy: {test_accuracy * 100:.2f}%')
    print(f'Test AUC (OvR): {test_auc:.4f}')
    print('Test Confusion Matrix:\n', confusion_matrix(y_test, y_test_pred))
    print('Test Classification Report:\n', classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

# === 1. Original (No Resampling)
train_and_evaluate(X_train, y_train, X_val, y_val, X_test, y_test, "Original (No Resampling)")

# === 2. SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
train_and_evaluate(X_train_smote, y_train_smote, X_val, y_val, X_test, y_test, "SMOTE (Over-sampling)")

# === 3. RUS
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
train_and_evaluate(X_train_rus, y_train_rus, X_val, y_val, X_test, y_test, "Random Under-Sampling (RUS)")
